In [2]:
"""
Adversarial Validation for PrimeVul Scenario 5 (Leave-One-CWE-Out)
------------------------------------------------------------------
Source domain: training pool (all CWEs except held-out) + 80% benign
Target domain: held-out CWE vulnerable instances + 20% benign
"""

import os
import gc
import json
import time
import traceback
import numpy as np
from tqdm import tqdm
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE, ADASYN, SVMSMOTE
import warnings

warnings.filterwarnings('ignore')


COMBINED_FILE = "../../../../../embedding/primevul/codebert/primevul_embedded.jsonl"
RAW_FILE      = "../../../../../dataset/primevul/primevul_raw_combined.jsonl"
EMB_KEY       = "emb"

RESULTS_PATH = "results/adversarial_validation_primevul_loco.json"
CACHE_DIR    = "results/oversample_cache_loco"
SEED         = 42
N_SPLITS     = 5

HELD_OUT_CWES = ["CWE-787"]
METHODS       = ["No Oversampling", "SMOTE", "ADASYN", "SVMSMOTE"]

os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
np.random.seed(SEED)

TOTAL_RUNS = len(HELD_OUT_CWES) * len(METHODS)


def load_results():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, "r") as f:
            return json.load(f)
    return {}


def save_results(results):
    tmp_path = RESULTS_PATH + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(results, f, indent=2)
    os.replace(tmp_path, RESULTS_PATH)


def result_key(cwe, method):
    return f"{cwe}__{method}"


def print_progress_summary(results):
    done = sum(1 for v in results.values() if v.get("status") == "ok")
    failed = sum(1 for v in results.values() if v.get("status") == "failed")
    remaining = TOTAL_RUNS - done - failed
    pct = 100 * (done + failed) / TOTAL_RUNS
    print(f"----- PROGRESS: {done} ok | {failed} failed | {remaining} remaining "
          f"| {pct:.1f}% of {TOTAL_RUNS} total runs -----")


def load_cwe_lookup(raw_path):
    cwe_map = {}
    with open(raw_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            cwe_list = rec.get("cwe", [])
            cwe_val = cwe_list[0] if cwe_list else None
            cwe_map[rec["idx"]] = cwe_val
    return cwe_map


def load_all_embeddings_with_cwe(path, cwe_map):
    t0 = time.time()
    X, y, cwe = [], [], []
    with open(path, "r") as f:
        lines = f.readlines()
    for line in tqdm(lines, desc="Parsing JSONL", unit="rec"):
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        X.append(rec[EMB_KEY])
        y.append(rec["target"])
        cwe.append(cwe_map.get(rec["idx"], None))
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int32)
    cwe = np.array(cwe, dtype=object)
    print(f"Loaded {X.shape[0]} embeddings in {time.time() - t0:.1f}s")
    return X, y, cwe


def build_loco_split(X_all, y_all, cwe_all, held_out_cwe):
    vuln_mask = (y_all == 1)
    test_vuln_mask = vuln_mask & (cwe_all == held_out_cwe)
    train_vuln_mask = vuln_mask & (cwe_all != held_out_cwe) & (cwe_all != None)

    if test_vuln_mask.sum() == 0:
        raise ValueError(f"No vulnerable instances found for {held_out_cwe}.")

    benign_idx = np.where(y_all == 0)[0]
    benign_train_idx, benign_test_idx = train_test_split(
        benign_idx, test_size=0.2, random_state=SEED
    )

    train_idx = np.concatenate([np.where(train_vuln_mask)[0], benign_train_idx])
    test_idx = np.concatenate([np.where(test_vuln_mask)[0], benign_test_idx])

    source_emb = X_all[train_idx]
    source_lbl = y_all[train_idx]
    target_emb = X_all[test_idx]
    target_lbl = y_all[test_idx]

    return source_emb, source_lbl, target_emb, target_lbl


def get_oversampled_source(cwe, method, source_emb, source_lbl):
    if method == "No Oversampling":
        return source_emb

    cache_path = os.path.join(CACHE_DIR, f"{cwe}_{method}.npy")
    if os.path.exists(cache_path):
        return np.load(cache_path)

    if method == "SMOTE":
        sampler = SMOTE(random_state=SEED)
    elif method == "ADASYN":
        sampler = ADASYN(random_state=SEED)
    elif method == "SVMSMOTE":
        sampler = SVMSMOTE(random_state=SEED)
    else:
        raise ValueError(f"Unknown method: {method}")

    resampled_emb, _ = sampler.fit_resample(source_emb, source_lbl)
    resampled_emb = resampled_emb.astype(np.float32)
    np.save(cache_path, resampled_emb)
    return resampled_emb


def adversarial_validation(source_emb, target_emb, label="", n_splits=N_SPLITS):
    X = np.vstack([source_emb, target_emb])
    y = np.array([0] * len(source_emb) + [1] * len(target_emb))

    print(f"    Fitting on {X.shape[0]} rows...")

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    auc_scores = []
    fold_bar = tqdm(cv.split(X, y), total=n_splits,
                     desc=f"    [{label}] CV folds", leave=True,
                     bar_format="{l_bar}{bar:25}{r_bar}")
    for train_idx, val_idx in fold_bar:
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        clf = HistGradientBoostingClassifier(max_iter=100, max_depth=3, random_state=SEED)
        clf.fit(X_train, y_train)
        proba = clf.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, proba)
        auc_scores.append(fold_auc)
        fold_bar.set_postfix({"fold_AUC": f"{fold_auc:.4f}"})
        del clf

    del X, y
    gc.collect()

    return float(np.mean(auc_scores)), float(np.std(auc_scores))


def main():
    results = load_results()
    print(f"Loaded {len(results)} existing results from {RESULTS_PATH}")
    print_progress_summary(results)

    cwe_map = load_cwe_lookup(RAW_FILE)
    X_all, y_all, cwe_all = load_all_embeddings_with_cwe(COMBINED_FILE, cwe_map)

    n_missing = int(np.sum((y_all == 1) & (cwe_all == None)))
    if n_missing > 0:
        print(f"WARNING: {n_missing} vulnerable rows had no CWE match.")

    n_cwes = len(HELD_OUT_CWES)
    for c_idx, held_out_cwe in enumerate(HELD_OUT_CWES, start=1):
        print(f"\n=== CWE {c_idx}/{n_cwes}: {held_out_cwe} ===")

        if all(result_key(held_out_cwe, m) in results for m in METHODS):
            print(f"  all methods already computed -- skipping CWE.")
            continue

        try:
            source_emb, source_lbl, target_emb, target_lbl = build_loco_split(
                X_all, y_all, cwe_all, held_out_cwe
            )

            scaler = StandardScaler()
            source_emb_scaled = scaler.fit_transform(source_emb).astype(np.float32)
            target_emb_scaled = scaler.transform(target_emb).astype(np.float32)

            print(f"  Source: {source_emb_scaled.shape} (vuln={int(source_lbl.sum())}) "
                  f"| Target: {target_emb_scaled.shape} (vuln={int(target_lbl.sum())})")

            del source_emb, target_emb
            gc.collect()

        except Exception as e:
            print(f"  [FATAL for {held_out_cwe}] Could not build split: {e}")
            traceback.print_exc()
            continue

        n_methods = len(METHODS)
        for m_idx, method in enumerate(METHODS, start=1):
            key = result_key(held_out_cwe, method)

            if key in results:
                print(f"  Method {m_idx}/{n_methods} [{method}]: already computed "
                      f"(AUC={results[key].get('mean_auc', 'N/A')}) -- skipping.")
                continue

            print(f"  Method {m_idx}/{n_methods} [{method}]:")
            t0 = time.time()
            resampled_source = None

            try:
                resampled_source = get_oversampled_source(
                    held_out_cwe, method, source_emb_scaled, source_lbl
                )

                mean_auc, std_auc = adversarial_validation(
                    resampled_source, target_emb_scaled, label=f"{held_out_cwe}/{method}"
                )

                results[key] = {
                    "held_out_cwe": held_out_cwe,
                    "method": method,
                    "n_source": int(resampled_source.shape[0]),
                    "n_target": int(target_emb_scaled.shape[0]),
                    "mean_auc": round(mean_auc, 4),
                    "std_auc": round(std_auc, 4),
                    "elapsed_sec": round(time.time() - t0, 1),
                    "status": "ok",
                }
                save_results(results)
                print(f"    AUC: {mean_auc:.4f} +/- {std_auc:.4f} "
                      f"({time.time() - t0:.1f}s) -- saved.")
                print_progress_summary(results)

            except Exception as e:
                print(f"    FAILED: {e}")
                traceback.print_exc()
                results[key] = {
                    "held_out_cwe": held_out_cwe,
                    "method": method,
                    "status": "failed",
                    "error": str(e),
                    "elapsed_sec": round(time.time() - t0, 1),
                }
                save_results(results)
                print_progress_summary(results)

            finally:
                if resampled_source is not None:
                    del resampled_source
                gc.collect()

        del source_emb_scaled, target_emb_scaled, source_lbl, target_lbl
        gc.collect()

    print("\nALL CWES AND METHODS PROCESSED (OR ALREADY CACHED).")
    print_progress_summary(results)
    print(f"Final results saved to {RESULTS_PATH}")


if __name__ == "__main__":
    main()

Loaded 0 existing results from results/adversarial_validation_primevul_loco.json
----- PROGRESS: 0 ok | 0 failed | 4 remaining | 0.0% of 4 total runs -----


Parsing JSONL: 100%|█████████████████████████| 224533/224533 [00:27<00:00, 8187.23rec/s]


Loaded 224533 embeddings in 36.9s

=== CWE 1/1: CWE-787 ===
  Source: (180323, 768) (vuln=5500) | Target: (44210, 768) (vuln=504)
  Method 1/4 [No Oversampling]:
    Fitting on 224533 rows...


    [CWE-787/No Oversampling] CV folds: 100%|█████████████████████████| 5/5 [00:13<00:00


    AUC: 0.5041 +/- 0.0020 (13.8s) -- saved.
----- PROGRESS: 1 ok | 0 failed | 3 remaining | 25.0% of 4 total runs -----
  Method 2/4 [SMOTE]:
    Fitting on 393856 rows...


...

    AUC: 0.6825 +/- 0.0029 (50.2s) -- saved.
----- PROGRESS: 2 ok | 0 failed | 2 remaining | 50.0% of 4 total runs -----
  Method 3/4 [ADASYN]:
    Fitting on 393083 rows...


    [CWE-787/ADASYN] CV folds: 100%|█████████████████████████| 5/5 [00:48<00:00,  9.65s/


    AUC: 0.6771 +/- 0.0026 (53.5s) -- saved.
----- PROGRESS: 3 ok | 0 failed | 1 remaining | 75.0% of 4 total runs -----
  Method 4/4 [SVMSMOTE]:
    Fitting on 393856 rows...


    [CWE-787/SVMSMOTE] CV folds: 100%|█████████████████████████| 5/5 [00:51<00:00, 10.34


    AUC: 0.7172 +/- 0.0025 (8176.6s) -- saved.
----- PROGRESS: 4 ok | 0 failed | 0 remaining | 100.0% of 4 total runs -----

ALL CWES AND METHODS PROCESSED (OR ALREADY CACHED).
----- PROGRESS: 4 ok | 0 failed | 0 remaining | 100.0% of 4 total runs -----
Final results saved to results/adversarial_validation_primevul_loco.json


In [4]:
"""
Adversarial Validation for PrimeVul Scenario 5 (Leave-One-CWE-Out) -- GAN Only
--------------------------------------------------------------------------------
Source domain: training pool (all CWEs except held-out) + 80% benign, GAN-oversampled
Target domain: held-out CWE vulnerable instances + 20% benign
"""

import os
import gc
import json
import time
import traceback
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings('ignore')


COMBINED_FILE = "../../../../../embedding/primevul/codebert/primevul_embedded.jsonl"
RAW_FILE      = "../../../../../dataset/primevul/primevul_raw_combined.jsonl"
EMB_KEY       = "emb"

RESULTS_PATH = "results/adversarial_validation_primevul_loco.json"
CACHE_DIR    = "results/oversample_cache_loco"
SEED         = 42
N_SPLITS     = 5

HELD_OUT_CWES = ["CWE-787"]
METHOD        = "GAN"

GAN_LATENT_DIM  = 100
GAN_EPOCHS      = 1500
GAN_BATCH_SIZE  = 64
GAN_LOG_EVERY   = 500

os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"

TOTAL_RUNS = len(HELD_OUT_CWES)


class Generator(nn.Module):
    def __init__(self, latent_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


def load_results():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, "r") as f:
            return json.load(f)
    return {}


def save_results(results):
    tmp_path = RESULTS_PATH + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(results, f, indent=2)
    os.replace(tmp_path, RESULTS_PATH)


def result_key(cwe, method):
    return f"{cwe}__{method}"


def print_progress_summary(results):
    done = sum(1 for k, v in results.items() if k.endswith(f"__{METHOD}") and v.get("status") == "ok")
    failed = sum(1 for k, v in results.items() if k.endswith(f"__{METHOD}") and v.get("status") == "failed")
    remaining = TOTAL_RUNS - done - failed
    pct = 100 * (done + failed) / TOTAL_RUNS
    print(f"----- GAN PROGRESS: {done} ok | {failed} failed | {remaining} remaining "
          f"| {pct:.1f}% of {TOTAL_RUNS} CWEs -----")


def load_cwe_lookup(raw_path):
    cwe_map = {}
    with open(raw_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            cwe_list = rec.get("cwe", [])
            cwe_val = cwe_list[0] if cwe_list else None
            cwe_map[rec["idx"]] = cwe_val
    return cwe_map


def load_all_embeddings_with_cwe(path, cwe_map):
    t0 = time.time()
    X, y, cwe = [], [], []
    with open(path, "r") as f:
        lines = f.readlines()
    for line in tqdm(lines, desc="Parsing JSONL", unit="rec"):
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        X.append(rec[EMB_KEY])
        y.append(rec["target"])
        cwe.append(cwe_map.get(rec["idx"], None))
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int32)
    cwe = np.array(cwe, dtype=object)
    print(f"Loaded {X.shape[0]} embeddings in {time.time() - t0:.1f}s | Device: {DEVICE}")
    return X, y, cwe


def build_loco_split(X_all, y_all, cwe_all, held_out_cwe):
    vuln_mask = (y_all == 1)
    test_vuln_mask = vuln_mask & (cwe_all == held_out_cwe)
    train_vuln_mask = vuln_mask & (cwe_all != held_out_cwe) & (cwe_all != None)

    if test_vuln_mask.sum() == 0:
        raise ValueError(f"No vulnerable instances found for {held_out_cwe}.")

    benign_idx = np.where(y_all == 0)[0]
    benign_train_idx, benign_test_idx = train_test_split(
        benign_idx, test_size=0.2, random_state=SEED
    )

    train_idx = np.concatenate([np.where(train_vuln_mask)[0], benign_train_idx])
    test_idx = np.concatenate([np.where(test_vuln_mask)[0], benign_test_idx])

    source_emb = X_all[train_idx]
    source_lbl = y_all[train_idx]
    target_emb = X_all[test_idx]
    target_lbl = y_all[test_idx]

    return source_emb, source_lbl, target_emb, target_lbl


def train_gan_and_generate(held_out_cwe, source_emb, source_lbl):
    gan_cache_path = os.path.join(CACHE_DIR, f"{held_out_cwe}_GAN.npy")
    if os.path.exists(gan_cache_path):
        return np.load(gan_cache_path)

    input_dim = source_emb.shape[1]
    minority_emb = source_emb[source_lbl == 1].astype(np.float32)
    n_to_generate = int(np.sum(source_lbl == 0)) - int(np.sum(source_lbl == 1))

    print(f"    Minority samples: {minority_emb.shape[0]} | Target to generate: {n_to_generate}")

    G = Generator(GAN_LATENT_DIM, input_dim).to(DEVICE)
    D = Discriminator(input_dim).to(DEVICE)
    opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    minority_tensor = torch.tensor(minority_emb).to(DEVICE)
    dataset = TensorDataset(minority_tensor)
    loader = DataLoader(dataset, batch_size=GAN_BATCH_SIZE, shuffle=True)

    t0 = time.time()
    epoch_bar = tqdm(range(GAN_EPOCHS), desc=f"    [{held_out_cwe}] GAN training", leave=False)
    for epoch in epoch_bar:
        for (real_batch,) in loader:
            bs = real_batch.size(0)

            z = torch.randn(bs, GAN_LATENT_DIM).to(DEVICE)
            fake = G(z).detach()
            loss_D = criterion(D(real_batch), torch.ones(bs, 1).to(DEVICE)) + \
                     criterion(D(fake), torch.zeros(bs, 1).to(DEVICE))
            opt_D.zero_grad()
            loss_D.backward()
            opt_D.step()

            z = torch.randn(bs, GAN_LATENT_DIM).to(DEVICE)
            loss_G = criterion(D(G(z)), torch.ones(bs, 1).to(DEVICE))
            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()

        if (epoch + 1) % GAN_LOG_EVERY == 0:
            epoch_bar.set_postfix({"D_loss": f"{loss_D.item():.4f}", "G_loss": f"{loss_G.item():.4f}"})

    print(f"    GAN training complete in {time.time() - t0:.1f}s")

    G.eval()
    with torch.no_grad():
        z = torch.randn(n_to_generate, GAN_LATENT_DIM).to(DEVICE)
        synthetic = G(z).cpu().numpy()

    resampled_emb = np.vstack([source_emb, synthetic]).astype(np.float32)
    np.save(gan_cache_path, resampled_emb)

    del G, D, opt_G, opt_D, minority_tensor, dataset, loader
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()

    return resampled_emb


def adversarial_validation(source_emb, target_emb, label="", n_splits=N_SPLITS):
    X = np.vstack([source_emb, target_emb])
    y = np.array([0] * len(source_emb) + [1] * len(target_emb))

    print(f"    Fitting on {X.shape[0]} rows...")

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    auc_scores = []
    fold_bar = tqdm(cv.split(X, y), total=n_splits,
                     desc=f"    [{label}] CV folds", leave=True,
                     bar_format="{l_bar}{bar:25}{r_bar}")
    for train_idx, val_idx in fold_bar:
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        clf = HistGradientBoostingClassifier(max_iter=100, max_depth=3, random_state=SEED)
        clf.fit(X_train, y_train)
        proba = clf.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, proba)
        auc_scores.append(fold_auc)
        fold_bar.set_postfix({"fold_AUC": f"{fold_auc:.4f}"})
        del clf

    del X, y
    gc.collect()

    return float(np.mean(auc_scores)), float(np.std(auc_scores))


def main():
    results = load_results()
    print(f"Loaded {len(results)} existing results from {RESULTS_PATH}")
    print_progress_summary(results)

    cwe_map = load_cwe_lookup(RAW_FILE)
    X_all, y_all, cwe_all = load_all_embeddings_with_cwe(COMBINED_FILE, cwe_map)

    n_missing = int(np.sum((y_all == 1) & (cwe_all == None)))
    if n_missing > 0:
        print(f"WARNING: {n_missing} vulnerable rows had no CWE match.")

    n_cwes = len(HELD_OUT_CWES)
    for c_idx, held_out_cwe in enumerate(HELD_OUT_CWES, start=1):
        key = result_key(held_out_cwe, METHOD)
        print(f"\n=== CWE {c_idx}/{n_cwes}: {held_out_cwe} [{METHOD}] ===")

        if key in results and results[key].get("status") == "ok":
            print(f"  already computed (AUC={results[key].get('mean_auc', 'N/A')}) -- skipping.")
            continue

        t0 = time.time()
        resampled_source = None

        try:
            source_emb, source_lbl, target_emb, target_lbl = build_loco_split(
                X_all, y_all, cwe_all, held_out_cwe
            )

            scaler = StandardScaler()
            source_emb_scaled = scaler.fit_transform(source_emb).astype(np.float32)
            target_emb_scaled = scaler.transform(target_emb).astype(np.float32)

            print(f"  Source: {source_emb_scaled.shape} (vuln={int(source_lbl.sum())}) "
                  f"| Target: {target_emb_scaled.shape} (vuln={int(target_lbl.sum())})")

            resampled_source = train_gan_and_generate(held_out_cwe, source_emb_scaled, source_lbl)

            mean_auc, std_auc = adversarial_validation(
                resampled_source, target_emb_scaled, label=f"{held_out_cwe}/{METHOD}"
            )

            results[key] = {
                "held_out_cwe": held_out_cwe,
                "method": METHOD,
                "n_source": int(resampled_source.shape[0]),
                "n_target": int(target_emb_scaled.shape[0]),
                "mean_auc": round(mean_auc, 4),
                "std_auc": round(std_auc, 4),
                "elapsed_sec": round(time.time() - t0, 1),
                "status": "ok",
            }
            save_results(results)
            print(f"  AUC: {mean_auc:.4f} +/- {std_auc:.4f} "
                  f"({time.time() - t0:.1f}s) -- saved.")
            print_progress_summary(results)

        except Exception as e:
            print(f"  FAILED: {e}")
            traceback.print_exc()
            results[key] = {
                "held_out_cwe": held_out_cwe,
                "method": METHOD,
                "status": "failed",
                "error": str(e),
                "elapsed_sec": round(time.time() - t0, 1),
            }
            save_results(results)
            print_progress_summary(results)

        finally:
            if resampled_source is not None:
                del resampled_source
            gc.collect()

    print("\nALL CWES PROCESSED FOR GAN (OR ALREADY CACHED).")
    print_progress_summary(results)
    print(f"Final results saved to {RESULTS_PATH}")


if __name__ == "__main__":
    main()

Loaded 4 existing results from results/adversarial_validation_primevul_loco.json
----- GAN PROGRESS: 0 ok | 0 failed | 1 remaining | 0.0% of 1 CWEs -----


Parsing JSONL: 100%|████████████████████████| 224533/224533 [00:21<00:00, 10464.00rec/s]


Loaded 224533 embeddings in 30.0s | Device: mps

=== CWE 1/1: CWE-787 [GAN] ===
  Source: (180323, 768) (vuln=5500) | Target: (44210, 768) (vuln=504)
    Minority samples: 5500 | Target to generate: 169323


    GAN training complete in 1596.2s
    Fitting on 393856 rows...


    [CWE-787/GAN] CV folds: 100%|█████████████████████████| 5/5 [00:51<00:00, 10.20s/it,


  AUC: 0.7425 +/- 0.0029 (1650.6s) -- saved.
----- GAN PROGRESS: 1 ok | 0 failed | 0 remaining | 100.0% of 1 CWEs -----

ALL CWES PROCESSED FOR GAN (OR ALREADY CACHED).
----- GAN PROGRESS: 1 ok | 0 failed | 0 remaining | 100.0% of 1 CWEs -----
Final results saved to results/adversarial_validation_primevul_loco.json
